In [ ]:
# ============================================================
# EFFECT OF COEFFICIENT QUANTIZATION ON A FOURTH-ORDER IIR FILTER
# ============================================================
#
# This notebook demonstrates how finite coefficient word length
# can modify the pole locations and the behavior of an IIR filter.
#
# A fourth-order filter is used because higher-order filters are
# generally more sensitive to coefficient quantization than a
# single second-order section.
#
# The theoretical filter contains two complex-conjugate pole pairs:
#
#       p1,2 = r1 exp(±j theta1)
#
#       p3,4 = r2 exp(±j theta2)
#
# The first pair can be moved interactively using the sliders r1
# and theta1. The second pair is kept fixed.
#
# The denominator polynomial is
#
#       D(z) = 1 + a1 z^(-1) + a2 z^(-2)
#                + a3 z^(-3) + a4 z^(-4).
#
# The coefficients a1, a2, a3 and a4 are calculated from the
# theoretical pole positions with high numerical precision.
#
# In a finite-precision implementation, these exact coefficients
# are replaced by quantized coefficients
#
#       a1_q, a2_q, a3_q, a4_q.
#
# The actual poles of the implemented filter are therefore the
# roots of
#
#       Dq(z) = 1 + a1_q z^(-1) + a2_q z^(-2)
#                 + a3_q z^(-3) + a4_q z^(-4).
#
#
# ============================================================
# IMPORTANT PEDAGOGICAL POINT
# ============================================================
#
# THE POLES THEMSELVES ARE NOT QUANTIZED.
#
# The sequence of operations is:
#
#       theoretical poles
#               |
#               v
#       theoretical denominator coefficients
#               |
#               v
#       coefficient quantization
#               |
#               v
#       quantized denominator coefficients
#               |
#               v
#       roots of the quantized denominator
#               |
#               v
#       quantized pole locations
#
# Consequently, small coefficient errors may produce much larger
# pole displacements, especially in higher-order filters.
#
#
# ============================================================
# STABILITY
# ============================================================
#
# The theoretical filter is stable because all theoretical poles
# satisfy
#
#                       |pk| < 1.
#
# The quantized implementation is BIBO stable only if
#
#                       |pk_q| < 1
#
# for every quantized pole.
#
# Three situations are distinguished:
#
#       |pk_q| < 1       -> STABLE
#
#       |pk_q| ≈ 1       -> UNIT-CIRCLE / MARGINAL CASE
#
#       |pk_q| > 1       -> UNSTABLE
#
# A numerical tolerance is used so that tiny floating-point
# deviations around |p| = 1 are not incorrectly classified as
# genuine instability.
#
#
# ============================================================
# COEFFICIENT FORMAT
# ============================================================
#
# The denominator coefficients of this fourth-order example may
# have magnitudes greater than 4.
#
# To avoid confusing coefficient quantization with coefficient
# overflow or saturation, a fixed numerical range
#
#                       -8 <= a < 8
#
# is used.
#
# The total word length B contains:
#
#       1 sign bit
#       3 integer/magnitude bits
#       F = B - 4 fractional bits
#
# and therefore
#
#                       q = 2^(-F).
#
# The minimum word length used in this notebook is B = 4 bits.
#
#
# ============================================================
# HOW TO USE THE NOTEBOOK
# ============================================================
#
# A useful demonstration is obtained with the default values:
#
#       r1 = 0.955
#       theta1 = 5 degrees
#
# Begin with
#
#       B = 16 bits
#
# and progressively reduce the coefficient word length.
#
# The theoretical poles remain unchanged, while the quantized
# poles move because the denominator coefficients change.
#
# Around sufficiently low precision, the quantized filter may
# change from
#
#       STABLE
#
# to
#
#       UNIT-CIRCLE / MARGINAL
#
# and finally to
#
#       UNSTABLE.
#
# For example, for suitable low-bit cases, a quantized pole may
# obtain a radius close to 1.2 or even larger.
#
#
# ============================================================
# DISPLAY MODES
# ============================================================
#
# POLE-ZERO PLOT
#
# Shows the theoretical and quantized pole locations relative to
# the unit circle.
#
# MAGNITUDE RESPONSE
#
# Compares the theoretical and quantized magnitude responses.
#
# PHASE RESPONSE
#
# Compares the theoretical and quantized phase responses.
#
# IMPULSE RESPONSE
#
# Shows the time-domain consequence of coefficient quantization.
# A genuinely unstable quantized filter produces an impulse
# response that eventually grows instead of decaying.
#
# Magnitude and phase responses should not be used alone to
# determine the stability of the quantized filter. Stability must
# be determined from the pole locations; the impulse response
# provides a complementary time-domain indication.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Layout
from IPython.display import display


# ------------------------------------------------------------
# Fixed second pole pair
# ------------------------------------------------------------

R2 = 0.880

THETA2_DEG = 38.0


# ------------------------------------------------------------
# Coefficient quantization
# ------------------------------------------------------------

def quantize_coefficients(a, B):

    F = B - 4

    q = 2.0**(-F)

    a = np.asarray(a, dtype=float)

    aq = np.where(a >= 0, np.floor(a / q + 0.5) * q, np.ceil(a / q - 0.5) * q)

    aq = np.clip(aq, -8.0, 8.0 - q)

    return aq, q, F


# ------------------------------------------------------------
# Generate theoretical poles and denominator coefficients
# ------------------------------------------------------------

def build_theoretical_filter(r1, theta1_deg):

    theta1 = np.deg2rad(theta1_deg)

    theta2 = np.deg2rad(THETA2_DEG)

    p1 = r1 * np.exp(1j * theta1)

    p2 = r1 * np.exp(-1j * theta1)

    p3 = R2 * np.exp(1j * theta2)

    p4 = R2 * np.exp(-1j * theta2)

    poles = np.array([p1, p2, p3, p4])

    denominator = np.poly(poles)

    denominator = np.real_if_close(denominator).real

    return poles, denominator


# ------------------------------------------------------------
# Impulse response
# ------------------------------------------------------------

def impulse_response(a, N):

    order = len(a)

    x = np.zeros(N)

    x[0] = 1.0

    y = np.zeros(N)

    for n in range(N):

        feedback = 0.0

        for k in range(1, order + 1):

            if n - k >= 0:

                feedback += a[k - 1] * y[n - k]

        y[n] = x[n] - feedback

    return y


# ------------------------------------------------------------
# Safe frequency response
# ------------------------------------------------------------

def safe_frequency_response(a, w):

    ejw = np.exp(-1j * w)

    denominator = np.ones_like(ejw, dtype=complex)

    for k, ak in enumerate(a, start=1):

        denominator += ak * ejw**k

    H = np.full(denominator.shape, np.nan + 1j * np.nan, dtype=complex)

    tolerance = 1e-12

    valid = np.abs(denominator) > tolerance

    H[valid] = 1.0 / denominator[valid]

    return H


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.cq-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.cq-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.cq-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.cq-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.cq-info {
    font-size: 13px;
    line-height: 1.50;
}

.cq-label {
    display: inline-block;
    min-width: 220px;
    font-weight: bold;
    vertical-align: top;
}

.cq-value {
    font-size: 14px;
    font-weight: bold;
    color: #1b3a57;
}

.cq-coeff-grid {
    display: inline-grid;
    grid-template-columns: 170px 170px;
    column-gap: 18px;
    row-gap: 1px;
    vertical-align: top;
}

.cq-good {
    color: #176b34;
    font-weight: bold;
}

.cq-warning {
    color: #b8860b;
    font-weight: bold;
}

.cq-bad {
    color: #b00020;
    font-weight: bold;
}

.cq-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 5px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="cq-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Effect of Coefficient Quantization on a Fourth-Order IIR Filter
    </div>

</div>
""")


# ------------------------------------------------------------
# Description
# ------------------------------------------------------------

description_html = HTML("""
<div class="cq-root">

    <div class="cq-description">

        The theoretical filter is stable because all original poles lie inside
        the unit circle. Its denominator coefficients are then represented
        using a finite word length.<br>

        Start from <b>16 bits</b> and progressively reduce the coefficient
        precision. The theoretical poles remain fixed, while the poles of the
        quantized denominator move and may eventually cross the unit circle.<br>

        Use the <b>Display</b> control to inspect the pole-zero plot,
        magnitude response, phase response or impulse response.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(
    width='610px',
    min_width='610px',
    overflow='visible'
)


# ------------------------------------------------------------
# Main plotting function
# ------------------------------------------------------------

def plot_quantized_filter(B=16, r1=0.955, theta1_deg=5.0, Nimp=50, display_mode='Pole-zero plot'):


    # --------------------------------------------------------
    # Theoretical filter
    # --------------------------------------------------------

    theoretical_poles, denominator = build_theoretical_filter(r1, theta1_deg)

    a = denominator[1:]


    # --------------------------------------------------------
    # Quantized coefficients
    # --------------------------------------------------------

    aq, q, F = quantize_coefficients(a, B)


    # --------------------------------------------------------
    # Quantized poles
    # --------------------------------------------------------

    denominator_q = np.concatenate(([1.0], aq))

    quantized_poles = np.roots(denominator_q)


    # --------------------------------------------------------
    # Pole magnitudes
    # --------------------------------------------------------

    theoretical_radii = np.abs(theoretical_poles)

    quantized_radii = np.abs(quantized_poles)

    max_quantized_radius = np.max(quantized_radii)


    # --------------------------------------------------------
    # Stability classification
    # --------------------------------------------------------

    stability_tolerance = 1e-6

    if max_quantized_radius < 1.0 - stability_tolerance:

        stability_text = "<span class='cq-good'>STABLE</span>"

        stability_explanation = "All quantized poles lie strictly inside the unit circle."

    elif max_quantized_radius <= 1.0 + stability_tolerance:

        stability_text = "<span class='cq-warning'>UNIT-CIRCLE / MARGINAL CASE</span>"

        stability_explanation = "At least one quantized pole lies numerically on the unit circle."

    else:

        stability_text = "<span class='cq-bad'>UNSTABLE AFTER QUANTIZATION</span>"

        stability_explanation = "At least one quantized pole lies genuinely outside the unit circle."


    # --------------------------------------------------------
    # Pole displacement
    # --------------------------------------------------------

    remaining = list(quantized_poles)

    pole_displacements = []

    for p in theoretical_poles:

        distances = [abs(p - pq) for pq in remaining]

        index = int(np.argmin(distances))

        pole_displacements.append(distances[index])

        remaining.pop(index)

    maximum_pole_displacement = np.max(pole_displacements)


    # --------------------------------------------------------
    # Frequency responses
    # --------------------------------------------------------

    w = np.linspace(0.0, np.pi, 2048)

    H = safe_frequency_response(a, w)

    Hq = safe_frequency_response(aq, w)

    magnitude = np.abs(H)

    magnitude_q = np.abs(Hq)

    mag_db = 20.0 * np.log10(np.maximum(magnitude, 1e-12))

    magq_db = 20.0 * np.log10(np.maximum(magnitude_q, 1e-12))

    phase_deg = np.unwrap(np.angle(H)) * 180.0 / np.pi

    phaseq_deg = np.unwrap(np.angle(Hq)) * 180.0 / np.pi


    # --------------------------------------------------------
    # Maximum magnitude-response error
    # --------------------------------------------------------

    valid = np.isfinite(magnitude) & np.isfinite(magnitude_q)

    if np.any(valid):

        maximum_magnitude_error = np.max(np.abs(magnitude[valid] - magnitude_q[valid]))

    else:

        maximum_magnitude_error = np.nan


    # --------------------------------------------------------
    # Impulse responses
    # --------------------------------------------------------

    h = impulse_response(a, Nimp)

    hq = impulse_response(aq, Nimp)


    # --------------------------------------------------------
    # Coefficient display
    # --------------------------------------------------------

    coefficients_text = f"""
    <div class="cq-coeff-grid">
        <div>a₁ = {a[0]:+.6f}</div>
        <div>a₂ = {a[1]:+.6f}</div>
        <div>a₃ = {a[2]:+.6f}</div>
        <div>a₄ = {a[3]:+.6f}</div>
    </div>
    """


    quantized_coefficients_text = f"""
    <div class="cq-coeff-grid">
        <div>a₁ᵠ = {aq[0]:+.6f}</div>
        <div>a₂ᵠ = {aq[1]:+.6f}</div>
        <div>a₃ᵠ = {aq[2]:+.6f}</div>
        <div>a₄ᵠ = {aq[3]:+.6f}</div>
    </div>
    """


    quantized_radii_text = ", ".join(
        [
            f"{radius:.6f}"
            for radius in np.sort(quantized_radii)
        ]
    )


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary_html.value = f"""
    <div class="cq-box">

        <div class="cq-title">
            Current Filter Data
        </div>

        <div class="cq-info">

            <span class="cq-label">Coefficient word length</span>
            B = <span class="cq-value">{B} bits</span>
            <br>

            <span class="cq-label">Fractional bits</span>
            F = {F}
            <br>

            <span class="cq-label">Quantization step</span>
            q = {q:.8f}
            <br>

            <span class="cq-label">Variable pole pair</span>
            r₁ = {r1:.4f}, &nbsp; θ₁ = {theta1_deg:.1f}°
            <br>

            <span class="cq-label">Fixed pole pair</span>
            r₂ = {R2:.4f}, &nbsp; θ₂ = {THETA2_DEG:.1f}°
            <br>

            <span class="cq-label">Theoretical coefficients</span>
            {coefficients_text}
            <br>

            <span class="cq-label">Quantized coefficients</span>
            {quantized_coefficients_text}
            <br>

            <span class="cq-label">Largest theoretical radius</span>
            {np.max(theoretical_radii):.6f}
            <br>

            <span class="cq-label">Quantized pole radii</span>
            {quantized_radii_text}
            <br>

            <span class="cq-label">Largest quantized radius</span>
            <span class="cq-value">{max_quantized_radius:.6f}</span>
            <br>

            <span class="cq-label">Maximum pole displacement</span>
            {maximum_pole_displacement:.6e}
            <br>

            <span class="cq-label">Maximum magnitude error</span>
            {maximum_magnitude_error:.6e}
            <br>

            <span class="cq-label">Quantized filter status</span>
            {stability_text}

        </div>

        <div class="cq-note">
            {stability_explanation}
            Pole displacement results from quantization of the denominator
            coefficients, not from direct quantization of the poles.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Create one single figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12.5, 4.0)
    )


    # ========================================================
    # POLE-ZERO PLOT
    # ========================================================

    if display_mode == 'Pole-zero plot':

        phi = np.linspace(0.0, 2.0 * np.pi, 600)


        ax.plot(
            np.cos(phi),
            np.sin(phi),
            'k--',
            linewidth=1.2,
            label='Unit circle'
        )


        ax.plot(
            np.real(theoretical_poles),
            np.imag(theoretical_poles),
            'bo',
            markersize=8,
            label='Theoretical poles'
        )


        ax.plot(
            np.real(quantized_poles),
            np.imag(quantized_poles),
            'rx',
            markersize=10,
            markeredgewidth=2.2,
            label='Quantized poles'
        )


        ax.plot(
            0.0,
            0.0,
            'go',
            markersize=8,
            markerfacecolor='none',
            markeredgewidth=1.7,
            label='Zeros at origin'
        )


        ax.axhline(
            0.0,
            linewidth=0.8
        )


        ax.axvline(
            0.0,
            linewidth=0.8
        )


        ax.set_xlim(
            -1.45,
            1.45
        )


        ax.set_ylim(
            -1.45,
            1.45
        )


        ax.set_aspect(
            'equal',
            adjustable='box'
        )


        ax.set_xlabel(
            'Real part'
        )


        ax.set_ylabel(
            'Imaginary part'
        )


        ax.set_title(
            'Pole Migration Due to Coefficient Quantization',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=4,
            fontsize=8,
            frameon=False
        )


    # ========================================================
    # MAGNITUDE RESPONSE
    # ========================================================

    elif display_mode == 'Magnitude response':

        ax.plot(
            w,
            mag_db,
            linewidth=1.8,
            label='Theoretical'
        )


        ax.plot(
            w,
            magq_db,
            '--',
            linewidth=1.8,
            label='Quantized'
        )


        ax.set_xlim(
            0.0,
            np.pi
        )


        ax.set_xticks(
            [
                0.0,
                np.pi / 2.0,
                np.pi
            ]
        )


        ax.set_xticklabels(
            [
                '0',
                'π/2',
                'π'
            ]
        )


        ax.set_xlabel(
            'Frequency ω'
        )


        ax.set_ylabel(
            'Magnitude [dB]'
        )


        ax.set_title(
            'Magnitude Response',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            fontsize=9,
            frameon=False
        )


    # ========================================================
    # PHASE RESPONSE
    # ========================================================

    elif display_mode == 'Phase response':

        ax.plot(
            w,
            phase_deg,
            linewidth=1.8,
            label='Theoretical'
        )


        ax.plot(
            w,
            phaseq_deg,
            '--',
            linewidth=1.8,
            label='Quantized'
        )


        ax.set_xlim(
            0.0,
            np.pi
        )


        ax.set_xticks(
            [
                0.0,
                np.pi / 2.0,
                np.pi
            ]
        )


        ax.set_xticklabels(
            [
                '0',
                'π/2',
                'π'
            ]
        )


        ax.set_xlabel(
            'Frequency ω'
        )


        ax.set_ylabel(
            'Phase [deg]'
        )


        ax.set_title(
            'Phase Response',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            fontsize=9,
            frameon=False
        )


    # ========================================================
    # IMPULSE RESPONSE
    # ========================================================

    else:

        n = np.arange(Nimp)


        ax.plot(
            n,
            h,
            linewidth=1.8,
            label='Theoretical'
        )


        ax.plot(
            n,
            hq,
            '--',
            linewidth=1.8,
            label='Quantized'
        )


        ax.axhline(
            0.0,
            linewidth=0.8
        )


        ax.set_xlim(
            0,
            Nimp - 1
        )


        ax.set_xlabel(
            'Sample index n'
        )


        ax.set_ylabel(
            'Amplitude'
        )


        ax.set_title(
            'Impulse Response',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            fontsize=9,
            frameon=False
        )


        if max_quantized_radius > 1.0 + stability_tolerance:

            ax.text(
                0.03,
                0.90,
                f'UNSTABLE: max |pᵠ| = {max_quantized_radius:.3f}',
                transform=ax.transAxes,
                fontsize=11,
                fontweight='bold',
                color='red',
                bbox=dict(
                    boxstyle='round',
                    facecolor='white',
                    alpha=0.90
                )
            )


        elif max_quantized_radius >= 1.0 - stability_tolerance:

            ax.text(
                0.03,
                0.90,
                'UNIT-CIRCLE / MARGINAL CASE',
                transform=ax.transAxes,
                fontsize=11,
                fontweight='bold',
                color='darkgoldenrod',
                bbox=dict(
                    boxstyle='round',
                    facecolor='white',
                    alpha=0.90
                )
            )


    # --------------------------------------------------------
    # Final figure spacing
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.90,
        bottom=0.25
    )


    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(
    width='310px'
)


slider_style = {
    'description_width': '125px'
}


bits_slider = IntSlider(
    value=16,
    min=4,
    max=16,
    step=1,
    description='Coeff. bits:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


r_slider = FloatSlider(
    value=0.955,
    min=0.75,
    max=0.995,
    step=0.005,
    description='Pole radius r₁:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.3f'
)


theta_slider = FloatSlider(
    value=5.0,
    min=5.0,
    max=70.0,
    step=0.5,
    description='Pole angle θ₁:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.1f'
)


impulse_slider = IntSlider(
    value=50,
    min=20,
    max=100,
    step=5,
    description='Impulse length:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


display_selector = RadioButtons(
    options=[
        'Pole-zero plot',
        'Magnitude response',
        'Phase response',
        'Impulse response'
    ],
    value='Pole-zero plot',
    description='Display:',
    style={'description_width': '70px'},
    layout=Layout(
        width='310px'
    )
)


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_quantized_filter,
    B=bits_slider,
    r1=r_slider,
    theta1_deg=theta_slider,
    Nimp=impulse_slider,
    display_mode=display_selector
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls = VBox(
    [
        HTML("<div class='cq-title'>Controls</div>"),
        bits_slider,
        r_slider,
        theta_slider,
        impulse_slider,
        HTML("<div style='height:5px;'></div>"),
        display_selector
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final notebook layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)